In [2]:
!pip install transformers datasets hf_transfer
!pip install 'accelerate>=0.26.0' matplotlib

In [1]:
# ==============================================================
# INTENT DETECTION: IndoBERT vs RoBERTa-base + Ensemble MLP (FIX)
# ==============================================================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
from torch import nn

# 1) Load dataset
ds = load_dataset("json", data_files={
   "train": "/workspace/train_anomaly_general_expanded (1).jsonl",
    "validation": "/workspace/valid_anomaly_general_expanded.jsonl"
})

labels = sorted(list({ex["label"] for ex in ds["train"]}))
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}
NUM_LABELS = len(labels)

def preprocess(examples, tokenizer):
    enc = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",   # penting
        max_length=128
    )
    # ubah label string -> integer
    enc["labels"] = [label2id[l] for l in examples["label"]]
    return enc

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        "acc": accuracy_score(p.label_ids, preds),
        "f1_macro": f1_score(p.label_ids, preds, average="macro")
    }

# 2) IndoBERT
indo_name = "indobenchmark/indobert-base-p1"
tok_indo = AutoTokenizer.from_pretrained(indo_name)
ds_indo = ds.map(lambda x: preprocess(x, tok_indo),
                 batched=True,
                 remove_columns=["text", "label"])  # <— FIX penting
model_indo = AutoModelForSequenceClassification.from_pretrained(
    indo_name, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
)

args_indo = TrainingArguments(
    output_dir="runs/indobert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro"
)

trainer_indo = Trainer(
    model=model_indo,
    args=args_indo,
    train_dataset=ds_indo["train"],
    eval_dataset=ds_indo["validation"],
    tokenizer=tok_indo,
    compute_metrics=compute_metrics
)

trainer_indo.train()
indo_eval = trainer_indo.evaluate()
print("\n✅ IndoBERT Evaluation:", indo_eval)

# 3) RoBERTa-base (default HF)
roberta_name = "roberta-base"
tok_roberta = AutoTokenizer.from_pretrained(roberta_name)
ds_roberta = ds.map(lambda x: preprocess(x, tok_roberta),
                    batched=True,
                    remove_columns=["text", "label"])  # <— FIX penting
model_roberta = AutoModelForSequenceClassification.from_pretrained(
    roberta_name, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
)

args_roberta = TrainingArguments(
    output_dir="runs/roberta_base",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro"
)

trainer_roberta = Trainer(
    model=model_roberta,
    args=args_roberta,
    train_dataset=ds_roberta["train"],
    eval_dataset=ds_roberta["validation"],
    tokenizer=tok_roberta,
    compute_metrics=compute_metrics
)

trainer_roberta.train()
roberta_eval = trainer_roberta.evaluate()
print("\n✅ RoBERTa (default) Evaluation:", roberta_eval)

# 4) Ekstrak logits validasi dari kedua model (untuk ensemble)
# def extract_logits(model, dataset):  # tokenizer tak dibutuhkan karena sudah tensorized
#     dl = torch.utils.data.DataLoader(dataset.with_format("torch"), batch_size=32)
#     logits_list, labels_list = [], []
#     model.eval()
#     for batch in dl:
#         with torch.no_grad():
#             out = model(input_ids=batch["input_ids"],
#                         attention_mask=batch["attention_mask"])
#         logits_list.append(out.logits)
#         labels_list.append(batch["labels"])
#     return torch.cat(logits_list), torch.cat(labels_list)

# logits_indo_val, labels_val = extract_logits(model_indo, ds_indo["validation"])
# logits_roberta_val, _ = extract_logits(model_roberta, ds_roberta["validation"])
def extract_logits(model, dataset):
    # pastikan model sudah di device yang benar
    device = next(model.parameters()).device
    dl = torch.utils.data.DataLoader(dataset.with_format("torch"), batch_size=32)

    logits_list, labels_list = [], []
    model.eval()
    for batch in dl:
        # pindahkan ONLY yang dibutuhkan ke device
        inputs = {
            "input_ids": batch["input_ids"].to(device),
            "attention_mask": batch["attention_mask"].to(device),
        }
        # token_type_ids mungkin tidak ada (RoBERTa), jadi cek dulu
        if "token_type_ids" in batch:
            inputs["token_type_ids"] = batch["token_type_ids"].to(device)

        with torch.no_grad():
            out = model(**inputs)

        # simpan logits di CPU supaya aman untuk pemrosesan berikutnya
        logits_list.append(out.logits.detach().cpu())
        labels_list.append(batch["labels"])  # biarkan di CPU

    return torch.cat(logits_list), torch.cat(labels_list)
logits_indo_val, labels_val = extract_logits(model_indo, ds_indo["validation"])
logits_roberta_val, _ = extract_logits(model_roberta, ds_roberta["validation"])

X_val = torch.cat([logits_indo_val, logits_roberta_val], dim=1)
y_val = labels_val


# 5) Ensemble MLP
class EnsembleMLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, out_dim)
        )
    def forward(self, x): return self.net(x)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ensemble = EnsembleMLP(X_val.shape[1], NUM_LABELS).to(dev)
X_val = X_val.to(dev).float()
y_val = y_val.to(dev).long()

# ensemble = EnsembleMLP(X_val.shape[1], NUM_LABELS)
opt = torch.optim.Adam(ensemble.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

# pakai split sederhana 80/20 pada validation sebagai meta-train/meta-test
n = int(0.8*len(X_val))
X_tr, X_te = X_val[:n], X_val[n:]
y_tr, y_te = y_val[:n], y_val[n:]



for _ in range(20):
    ensemble.train()
    opt.zero_grad()
    out = ensemble(X_tr)
    loss = crit(out, y_tr)
    loss.backward()
    opt.step()

ensemble.eval()
with torch.no_grad():
    pred = ensemble(X_te).argmax(dim=1)
y_te = y_te.detach().cpu()
pred = pred.detach().cpu()
acc = accuracy_score(y_te.numpy(), pred.numpy())
f1  = f1_score(y_te.numpy(), pred.numpy(), average="macro")

print(f"\n✅ Ensemble MLP Accuracy: {acc:.4f}, F1: {f1:.4f}")
print(f"IndoBERT acc={indo_eval['eval_acc']:.4f}, RoBERTa acc={roberta_eval['eval_acc']:.4f}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_9370/3141123151.py:63: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_indo = Trainer(


Epoch,Training Loss,Validation Loss,Acc,F1 Macro
1,No log,1.321925,0.538462,0.491304
2,No log,1.177725,0.528205,0.482293
3,No log,1.187964,0.553846,0.509472
4,No log,1.176567,0.543590,0.517500
5,No log,1.180201,0.528205,0.505839



✅ IndoBERT Evaluation: {'eval_loss': 1.1765666007995605, 'eval_acc': 0.5435897435897435, 'eval_f1_macro': 0.517500067444216, 'eval_runtime': 0.475, 'eval_samples_per_second': 410.528, 'eval_steps_per_second': 14.737, 'epoch': 5.0}


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_9370/3141123151.py:98: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_roberta = Trainer(


Epoch,Training Loss,Validation Loss,Acc,F1 Macro
1,No log,2.018633,0.328205,0.247293
2,No log,1.389816,0.533333,0.479145
3,No log,1.240997,0.538462,0.481950
4,No log,1.232208,0.523077,0.466932
5,No log,1.211492,0.538462,0.502481



✅ RoBERTa (default) Evaluation: {'eval_loss': 1.2114924192428589, 'eval_acc': 0.5384615384615384, 'eval_f1_macro': 0.5024812360719407, 'eval_runtime': 0.4369, 'eval_samples_per_second': 446.302, 'eval_steps_per_second': 16.021, 'epoch': 5.0}

✅ Ensemble MLP Accuracy: 0.4615, F1: 0.3855
IndoBERT acc=0.5436, RoBERTa acc=0.5385


In [2]:
# ==============================================================
# INTENT DETECTION: IndoBERT vs RoBERTa-base + Ensemble MLP (FIX)
# ==============================================================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
from torch import nn

# 1) Load dataset
ds = load_dataset("json", data_files={
   "train": "/workspace/train_anomaly_general_expanded (1).jsonl",
    "validation": "/workspace/valid_anomaly_general_expanded.jsonl"
})

labels = sorted(list({ex["label"] for ex in ds["train"]}))
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}
NUM_LABELS = len(labels)

def preprocess(examples, tokenizer):
    enc = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",   # penting
        max_length=128
    )
    # ubah label string -> integer
    enc["labels"] = [label2id[l] for l in examples["label"]]
    return enc

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        "acc": accuracy_score(p.label_ids, preds),
        "f1_macro": f1_score(p.label_ids, preds, average="macro")
    }

# 2) IndoBERT
indo_name = "indobenchmark/indobert-base-p1"
tok_indo = AutoTokenizer.from_pretrained(indo_name)
ds_indo = ds.map(lambda x: preprocess(x, tok_indo),
                 batched=True,
                 remove_columns=["text", "label"])  # <— FIX penting
model_indo = AutoModelForSequenceClassification.from_pretrained(
    indo_name, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
)

args_indo = TrainingArguments(
    output_dir="runs/indobert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs=100,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro"
)

trainer_indo = Trainer(
    model=model_indo,
    args=args_indo,
    train_dataset=ds_indo["train"],
    eval_dataset=ds_indo["validation"],
    tokenizer=tok_indo,
    compute_metrics=compute_metrics
)

trainer_indo.train()
indo_eval = trainer_indo.evaluate()
print("\n✅ IndoBERT Evaluation:", indo_eval)

# 3) RoBERTa-base (default HF)
roberta_name = "roberta-base"
tok_roberta = AutoTokenizer.from_pretrained(roberta_name)
ds_roberta = ds.map(lambda x: preprocess(x, tok_roberta),
                    batched=True,
                    remove_columns=["text", "label"])  # <— FIX penting
model_roberta = AutoModelForSequenceClassification.from_pretrained(
    roberta_name, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
)

args_roberta = TrainingArguments(
    output_dir="runs/roberta_base",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs=100,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro"
)

trainer_roberta = Trainer(
    model=model_roberta,
    args=args_roberta,
    train_dataset=ds_roberta["train"],
    eval_dataset=ds_roberta["validation"],
    tokenizer=tok_roberta,
    compute_metrics=compute_metrics
)

trainer_roberta.train()
roberta_eval = trainer_roberta.evaluate()
print("\n✅ RoBERTa (default) Evaluation:", roberta_eval)

# 4) Ekstrak logits validasi dari kedua model (untuk ensemble)
# def extract_logits(model, dataset):  # tokenizer tak dibutuhkan karena sudah tensorized
#     dl = torch.utils.data.DataLoader(dataset.with_format("torch"), batch_size=32)
#     logits_list, labels_list = [], []
#     model.eval()
#     for batch in dl:
#         with torch.no_grad():
#             out = model(input_ids=batch["input_ids"],
#                         attention_mask=batch["attention_mask"])
#         logits_list.append(out.logits)
#         labels_list.append(batch["labels"])
#     return torch.cat(logits_list), torch.cat(labels_list)

# logits_indo_val, labels_val = extract_logits(model_indo, ds_indo["validation"])
# logits_roberta_val, _ = extract_logits(model_roberta, ds_roberta["validation"])
def extract_logits(model, dataset):
    # pastikan model sudah di device yang benar
    device = next(model.parameters()).device
    dl = torch.utils.data.DataLoader(dataset.with_format("torch"), batch_size=32)

    logits_list, labels_list = [], []
    model.eval()
    for batch in dl:
        # pindahkan ONLY yang dibutuhkan ke device
        inputs = {
            "input_ids": batch["input_ids"].to(device),
            "attention_mask": batch["attention_mask"].to(device),
        }
        # token_type_ids mungkin tidak ada (RoBERTa), jadi cek dulu
        if "token_type_ids" in batch:
            inputs["token_type_ids"] = batch["token_type_ids"].to(device)

        with torch.no_grad():
            out = model(**inputs)

        # simpan logits di CPU supaya aman untuk pemrosesan berikutnya
        logits_list.append(out.logits.detach().cpu())
        labels_list.append(batch["labels"])  # biarkan di CPU

    return torch.cat(logits_list), torch.cat(labels_list)
logits_indo_val, labels_val = extract_logits(model_indo, ds_indo["validation"])
logits_roberta_val, _ = extract_logits(model_roberta, ds_roberta["validation"])

X_val = torch.cat([logits_indo_val, logits_roberta_val], dim=1)
y_val = labels_val


# 5) Ensemble MLP
class EnsembleMLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, out_dim)
        )
    def forward(self, x): return self.net(x)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ensemble = EnsembleMLP(X_val.shape[1], NUM_LABELS).to(dev)
X_val = X_val.to(dev).float()
y_val = y_val.to(dev).long()

# ensemble = EnsembleMLP(X_val.shape[1], NUM_LABELS)
opt = torch.optim.Adam(ensemble.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

# pakai split sederhana 80/20 pada validation sebagai meta-train/meta-test
n = int(0.8*len(X_val))
X_tr, X_te = X_val[:n], X_val[n:]
y_tr, y_te = y_val[:n], y_val[n:]



for _ in range(20):
    ensemble.train()
    opt.zero_grad()
    out = ensemble(X_tr)
    loss = crit(out, y_tr)
    loss.backward()
    opt.step()

ensemble.eval()
with torch.no_grad():
    pred = ensemble(X_te).argmax(dim=1)
y_te = y_te.detach().cpu()
pred = pred.detach().cpu()
acc = accuracy_score(y_te.numpy(), pred.numpy())
f1  = f1_score(y_te.numpy(), pred.numpy(), average="macro")

print(f"\n✅ Ensemble MLP Accuracy: {acc:.4f}, F1: {f1:.4f}")
print(f"IndoBERT acc={indo_eval['eval_acc']:.4f}, RoBERTa acc={roberta_eval['eval_acc']:.4f}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_374/3379118760.py:63: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_indo = Trainer(


Epoch,Training Loss,Validation Loss,Acc,F1 Macro
1,No log,1.511684,0.456410,0.394969
2,No log,1.239105,0.538462,0.472587
3,No log,1.174572,0.548718,0.500695
4,No log,1.192726,0.512821,0.473992
5,No log,1.211132,0.497436,0.482249
6,No log,1.217413,0.523077,0.499359
7,No log,1.174986,0.543590,0.545883
8,No log,1.240831,0.512821,0.511924
9,1.263200,1.286362,0.528205,0.518745
10,1.263200,1.279971,0.538462,0.533953



✅ IndoBERT Evaluation: {'eval_loss': 1.174985647201538, 'eval_acc': 0.5435897435897435, 'eval_f1_macro': 0.5458829444479562, 'eval_runtime': 0.5009, 'eval_samples_per_second': 389.308, 'eval_steps_per_second': 13.975, 'epoch': 100.0}


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_374/3379118760.py:98: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_roberta = Trainer(


Epoch,Training Loss,Validation Loss,Acc,F1 Macro
1,No log,1.795211,0.435897,0.346902
2,No log,1.311609,0.548718,0.486147
3,No log,1.236603,0.543590,0.486543
4,No log,1.245813,0.517949,0.466579
5,No log,1.255789,0.507692,0.475738
6,No log,1.212435,0.528205,0.475527
7,No log,1.205239,0.528205,0.484314
8,No log,1.205177,0.523077,0.493143
9,1.393300,1.253484,0.543590,0.503466
10,1.393300,1.223564,0.553846,0.511075



✅ RoBERTa (default) Evaluation: {'eval_loss': 2.6217784881591797, 'eval_acc': 0.5487179487179488, 'eval_f1_macro': 0.5532125692343083, 'eval_runtime': 0.4394, 'eval_samples_per_second': 443.767, 'eval_steps_per_second': 15.93, 'epoch': 100.0}

✅ Ensemble MLP Accuracy: 0.5128, F1: 0.4509
IndoBERT acc=0.5436, RoBERTa acc=0.5487
